# Chat Models — Try it in PyTorch

An **optional** hands-on companion to [Chapter 10](https://learnai.robennals.org/chat). The chapter says a base model and a chat model are the same machine, and that everything between them is post-training. Here you run both and see for yourself.

New to PyTorch? The [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) is a quick introduction.

## About the models used here

The models in this notebook are small enough to run free, in your browser, in a few minutes. They are hundreds of times smaller than the ones behind ChatGPT or Claude, and it shows: they make mistakes a frontier model would not. What they do have is the same machinery. Everything here works the same way at a thousand times the size, which is the point.

We use **Qwen3-1.7B-Base** and **Qwen3-1.7B**. Same family, same pre-training, and the second has been through post-training. That is the only difference between them.

In [ ]:
!pip install -q transformers accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
print("running on", DEVICE)

def load(name):
    """Download a model and its tokenizer, and put it on the fastest chip we have."""
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to(DEVICE)
    return tokenizer, model

def complete(tokenizer, model, text, max_new_tokens=60):
    """Give the model a piece of text and return what it writes next.

    The small limits in this notebook are about how much to print, not about
    how much the model is allowed to say. A base model given free rein will
    happily continue for pages, which is itself the point being made.
    """
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


## What a raw model does

Two models, the same prompts. The base model has only ever been asked to predict the next word.

In [ ]:
base_tokenizer, base_model = load("Qwen/Qwen3-1.7B-Base")
print("base model loaded")

In [ ]:
prompts = [
    "How do I stop my sourdough starter from going mouldy?",
    "My laptop won't turn on. Can you help?",
    "Is it safe to reheat rice?",
]

for prompt in prompts:
    print("PROMPT:", prompt)
    print("BASE MODEL CONTINUES:", complete(base_tokenizer, base_model, prompt, 60))
    print("-" * 70)

Read those carefully, and be fair to the base model. Sometimes it answers, because pages that answer questions are part of what it read. What it does not do is behave like something you are talking to: it repeats the question back, or drifts into a second question, or carries on past the answer because a web page always has more page after it. Nothing has told it that a reply is a thing that ends.

Now the same prompts, to the model that has been post-trained.

In [ ]:
import gc

# Two 1.7B models fit alongside each other on a Colab GPU, but only just, and
# we are done with this one. Dropping the reference is not enough on its own:
# the memory is only handed back once the cache is emptied too.
del base_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

chat_tokenizer, chat_model = load("Qwen/Qwen3-1.7B")
print("chat model loaded")

In [ ]:
def chat(prompt, max_new_tokens=120, thinking=False):
    """Wrap the prompt in the chat template, then complete it."""
    text = chat_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    return complete(chat_tokenizer, chat_model, text, max_new_tokens)

for prompt in prompts:
    print("PROMPT:", prompt)
    print("CHAT MODEL ANSWERS:", chat(prompt, 90))
    print("-" * 70)

**Try your own.** Put your own prompts in the list above and run both cells again. Some things to try: a question with a clear factual answer, an instruction like "translate this to French", and something that reads like the middle of a document rather than a request. The base model does best on the last one, which is the whole point.

## Making it a conversation

The chapter says a chat is a prompt and a completion, with markers saying who is speaking. Here is the actual text the model is handed.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's the capital of Australia?"},
]

prompt_text = chat_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

print(repr(prompt_text))

Those `<|im_start|>` and `<|im_end|>` pieces are real tokens with their own numbers in the vocabulary, not punctuation the model has to interpret.

In [ ]:
for marker in ["<|im_start|>", "<|im_end|>"]:
    print(marker, "->", chat_tokenizer.encode(marker))

The prompt ends with the marker that opens the model's turn. That is what makes an answer the thing left to write.

## Repeated turns

The model keeps nothing between turns. Everything it needs has to be in the prompt, so the whole conversation goes back every time, including its own earlier replies.

In [ ]:
conversation = [{"role": "user", "content": "What's the capital of Australia?"}]

for turn in range(3):
    prompt_text = chat_tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    n_tokens = len(chat_tokenizer(prompt_text).input_ids)
    reply = complete(chat_tokenizer, chat_model, prompt_text, 40).strip()
    print(f"turn {turn + 1}: prompt is {n_tokens} tokens -> {reply[:70]!r}")
    conversation.append({"role": "assistant", "content": reply})
    conversation.append({"role": "user", "content": "Are you sure?"})

The prompt grows every turn. Nothing was remembered; it was re-read.

You may also spot the model getting a fact wrong somewhere in there. A 1.7B model does that. It is worth seeing rather than hiding: the machinery is the same as a frontier model's, and the size is what buys reliability.

Here is the third turn's prompt in full. The model's own first answer is in there, put back by the code above.

In [ ]:
print(chat_tokenizer.apply_chat_template(
    conversation[:5], tokenize=False, add_generation_prompt=True, enable_thinking=False))

## Learning from what users say

The chapter's post-training section describes a signal that costs nothing: users say when a reply missed. Detecting that is easy enough for a small model to do, which is why it scales.

In [ ]:
endings = [
    ("perfect, thanks, that's exactly what I needed", "happy"),
    ("you didn't answer my question", "unhappy"),
    ("great, that worked first time", "happy"),
    ("no, I asked about the other one", "unhappy"),
    ("brilliant, cheers", "happy"),
    ("this is wrong, the total should be 40", "unhappy"),
]

happy_token = chat_tokenizer.encode("happy")[0]
unhappy_token = chat_tokenizer.encode("un")[0]

def happiness(message):
    """How likely is the model to answer 'happy' rather than 'unhappy'?"""
    question = (f'A user said this to an assistant: "{message}"\n\n'
                "Was the user happy with the answer they got? "
                "Reply with one word, happy or unhappy.")
    text = chat_tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = chat_tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = chat_model(**inputs).logits[0, -1]
    probabilities = torch.softmax(logits.float(), dim=-1)
    return probabilities[happy_token].item(), probabilities[unhappy_token].item()

correct = 0
for message, truth in endings:
    p_happy, p_unhappy = happiness(message)
    guess = "happy" if p_happy > p_unhappy else "unhappy"
    correct += guess == truth
    print(f"happy {p_happy:.2f} / unhappy {p_unhappy:.2f}  ->  {guess:<8} {message[:42]}")

print(f"\n{correct} out of {len(endings)} right")

Rather than read the model's answer as words, we asked it for the probability of the next token being `happy` against `unhappy`. That is what the model actually produces, and the chapter's post-training section leans on it: a number you can do arithmetic with, rather than a sentence you have to interpret.

A model this small, reading one message with no other context, is confident and right. A chat product sees millions of these a day and they cost nothing to collect, which is what makes the signal so tempting, and why the chapter warns about what it really measures: whether the person was pleased, not whether the answer was right.

## What you saw

- The base model and the chat model know the same things. Only one of them answers you.
- A conversation is one stream of text with markers in it. The markers are ordinary tokens.
- Nothing is remembered between turns. The whole conversation is sent again each time.
- The cheapest post-training signal, noticing an unhappy user, is easy enough for a tiny model to read.